# LeetCode 628 — Maximum Product of Three Numbers bằng Python

**Link đề:** https://leetcode.com/problems/maximum-product-of-three-numbers/
**Độ khó:** Easy · **Chủ đề:** Array, Math, Sorting

```python
class Solution:
    def maximumProduct(self, nums: list[int]) -> int:
        ...
```

## Mục tiêu học
1. Hiểu vì sao "lấy 3 số lớn nhất" **chưa đủ**: hai số âm nhân nhau thành số dương lớn.
2. Phân tích dấu (âm/dương/0) để rút ra **chỉ có 2 ứng viên**: `3 số lớn nhất` và `2 số nhỏ nhất × số lớn nhất`.
3. Đi từ brute force `O(n³)` → **sắp xếp** `O(n log n)` → **một lượt giữ 3 max + 2 min** `O(n)`; biết thêm `heapq.nlargest/nsmallest`.
4. Luyện `sorted`, chỉ số âm `a[-1]`, `itertools.combinations`, `math.prod`, cập nhật "top-k" bằng chuỗi `if/elif`.
5. Trace, chứng minh, Time/Space, test đối chiếu, 6 bài tập (có tổng quát **k số**) với đáp án cuối.

## 1. Đề bài và ví dụ

Cho mảng số nguyên `nums`. Tìm **ba số** (ở ba vị trí khác nhau) có **tích lớn nhất** và trả về tích đó.

| nums | Output | Giải thích |
|---|---:|---|
| `[1,2,-3]` | `-6` | chỉ có đúng 3 số: `1·2·(-3)` |
| `[1,2,3,4]` | `24` | `2·3·4` |
| `[-1,-2,-3]` | `-6` | `(-1)(-2)(-3)` |

Thêm một ví dụ "bẫy": `[-10,-10,1,3,2]` → `300` = `(-10)(-10)·3`, **lớn hơn** `3·2·1 = 6`.

**Constraints:** `3 <= nums.length <= 10^4`; `-1000 <= nums[i] <= 1000`.

Tích lớn nhất có thể: `1000³ = 10^9` – Python số nguyên không tràn, nhưng ở C++/Java vẫn vừa `int`/`long`.

## 2. Nền tảng Python

### 2.1. Dấu của tích
- `(+)(+)(+) = +`, `(−)(−)(+) = +`, `(−)(+)(+) = −`, `(−)(−)(−) = −`.
- Hai số âm có **trị tuyệt đối lớn** nhân nhau ra số dương **rất lớn**.

In [1]:
print("(-10)*(-10)*3 =", (-10) * (-10) * 3)
print("3*2*1         =", 3 * 2 * 1)
print("(-1)*(-2)*(-3) =", (-1) * (-2) * (-3))

(-10)*(-10)*3 = 300
3*2*1         = 6
(-1)*(-2)*(-3) = -6


### 2.2. `sorted` và chỉ số âm
Sau `a = sorted(nums)` (tăng dần): `a[0], a[1]` là hai số **nhỏ nhất** (âm nhất), `a[-1], a[-2], a[-3]` là ba số **lớn nhất**. `a[-1]` nghĩa là phần tử cuối.

In [2]:
a = sorted([-10, -10, 1, 3, 2])
print("a =", a)
print("2 nhỏ nhất:", a[0], a[1], "| 3 lớn nhất:", a[-3], a[-2], a[-1])

a = [-10, -10, 1, 2, 3]
2 nhỏ nhất: -10 -10 | 3 lớn nhất: 1 2 3


### 2.3. `itertools.combinations` và `math.prod`
- `combinations(nums, 3)` sinh mọi bộ 3 phần tử ở **3 vị trí khác nhau** (không lặp, không quan tâm thứ tự).
- `math.prod(iterable)` nhân tất cả phần tử.

In [3]:
from itertools import combinations
import math

for combo in combinations([1, 2, 3, 4], 3):
    print(combo, "->", math.prod(combo))

(1, 2, 3) -> 6
(1, 2, 4) -> 8
(1, 3, 4) -> 12
(2, 3, 4) -> 24


### 2.4. Đếm số bộ ba để thấy vì sao brute force chậm
Số bộ ba từ `n` phần tử là `C(n, 3) = n(n−1)(n−2)/6`. `math.comb(n, k)` tính tổ hợp.

In [4]:
for n in (5, 100, 10_000):
    print(f"n = {n:>6}: số bộ ba = {math.comb(n, 3):,}")

n =      5: số bộ ba = 10
n =    100: số bộ ba = 161,700
n =  10000: số bộ ba = 166,616,670,000


## 3. Cách 1 — Brute force thử mọi bộ ba

Ba vòng lặp lồng nhau `i < j < k` (hoặc `combinations`). Đúng nhưng `O(n³)`: với `n = 10^4` là ~`1.7·10^11` bộ ba – không thể.

In [5]:
def max_product_bruteforce(nums):
    n = len(nums)
    best = -math.inf
    for i in range(n):
        for j in range(i + 1, n):
            for k in range(j + 1, n):
                best = max(best, nums[i] * nums[j] * nums[k])
    return best

def max_product_combinations(nums):
    return max(math.prod(c) for c in combinations(nums, 3))

for nums in ([1, 2, -3], [1, 2, 3, 4], [-1, -2, -3], [-10, -10, 1, 3, 2]):
    print(nums, "->", max_product_bruteforce(nums), max_product_combinations(nums))

[1, 2, -3] -> -6 -6
[1, 2, 3, 4] -> 24 24
[-1, -2, -3] -> -6 -6
[-10, -10, 1, 3, 2] -> 300 300


`math.inf` là vô cực; `-math.inf` nhỏ hơn mọi số – khởi tạo max an toàn (vì đáp án có thể âm, **không** được khởi tạo `best = 0`).

## 4. Insight: chỉ có hai ứng viên

Gọi mảng đã sắp xếp là `a`. Bộ ba tốt nhất **luôn** là một trong:
1. `a[-1] * a[-2] * a[-3]` – ba số lớn nhất;
2. `a[0] * a[1] * a[-1]` – hai số nhỏ nhất (thường là hai số âm "to") nhân số lớn nhất.

Phân tích theo dấu của tích tối ưu:
- **Tích tối ưu ≥ 0** (có thể dương): ta muốn trị tuyệt đối lớn với số lượng số âm **chẵn** (0 hoặc 2). Không âm nào → 3 số lớn nhất. Hai âm → hai âm phải có trị tuyệt đối lớn nhất = `a[0], a[1]`, số còn lại dương lớn nhất = `a[-1]`.
- **Tích tối ưu < 0** (bắt buộc âm, ví dụ toàn số âm hoặc chỉ có 3 số): muốn trị tuyệt đối **nhỏ** nhất → chọn ba số "gần 0" nhất phía lớn = `a[-1], a[-2], a[-3]`.

Vì sao không cần `a[0] * a[-1] * a[-2]` (đúng một số âm)? Khi có số không âm, tích một số âm là `<= 0`, mà luôn tồn tại một trong hai ứng viên `>= 0` nên nó không thể thắng. (Mục 9 chứng minh đầy đủ, mục 11 kiểm chứng bằng brute force.)

In [6]:
a = sorted([-10, -10, 1, 3, 2])
cand1 = a[-1] * a[-2] * a[-3]
cand2 = a[0] * a[1] * a[-1]
print("3 lớn nhất       :", cand1)
print("2 nhỏ nhất × max :", cand2)
print("Đáp án           :", max(cand1, cand2))

3 lớn nhất       : 6
2 nhỏ nhất × max : 300
Đáp án           : 300


## 5. Cách 2 — Sắp xếp rồi so hai ứng viên

Giảng từng đoạn:
1. `a = sorted(nums)` – `O(n log n)`.
2. Tính `cand1`, `cand2` như trên.
3. Trả `max(cand1, cand2)`.

In [7]:
def max_product_sort(nums):
    a = sorted(nums)
    return max(a[-1] * a[-2] * a[-3], a[0] * a[1] * a[-1])

for nums in ([1, 2, -3], [1, 2, 3, 4], [-1, -2, -3], [-10, -10, 1, 3, 2]):
    print(nums, "->", max_product_sort(nums))

[1, 2, -3] -> -6
[1, 2, 3, 4] -> 24
[-1, -2, -3] -> -6
[-10, -10, 1, 3, 2] -> 300


### 5.1. Bảng các tình huống dấu

| Mảng (đã sắp xếp) | 3 lớn nhất | 2 nhỏ × max | Đáp án | Nhận xét |
|---|---:|---:|---:|---|
| `[1,2,3,4]` | 24 | 4 | 24 | toàn dương |
| `[-10,-10,1,2,3]` | 6 | 300 | 300 | hai âm lớn thắng |
| `[-5,-4,-3,-2,-1]` | -6 | -20 | -6 | toàn âm: chọn gần 0 nhất |
| `[-4,-3,-2,0]` | 0 | 0 | 0 | có số 0 |
| `[-3,1,2]` | -6 | -6 | -6 | chỉ có 3 số |

In [8]:
def explain(nums):
    a = sorted(nums)
    c1, c2 = a[-1] * a[-2] * a[-3], a[0] * a[1] * a[-1]
    print(f"{str(a):<22} 3 lớn nhất={c1:>5} | 2 nhỏ×max={c2:>5} | đáp án={max(c1, c2):>5}")

for nums in ([1, 2, 3, 4], [-10, -10, 1, 2, 3], [-5, -4, -3, -2, -1], [-4, -3, -2, 0], [-3, 1, 2]):
    explain(nums)

[1, 2, 3, 4]           3 lớn nhất=   24 | 2 nhỏ×max=    8 | đáp án=   24
[-10, -10, 1, 2, 3]    3 lớn nhất=    6 | 2 nhỏ×max=  300 | đáp án=  300
[-5, -4, -3, -2, -1]   3 lớn nhất=   -6 | 2 nhỏ×max=  -20 | đáp án=   -6
[-4, -3, -2, 0]        3 lớn nhất=    0 | 2 nhỏ×max=    0 | đáp án=    0
[-3, 1, 2]             3 lớn nhất=   -6 | 2 nhỏ×max=   -6 | đáp án=   -6


## 6. Cách 3 — Một lượt: giữ 3 số lớn nhất và 2 số nhỏ nhất

Sắp xếp cả mảng là "thừa" – ta chỉ cần 5 số. Duyệt một lần và cập nhật:
- `max1 >= max2 >= max3` (ba số lớn nhất);
- `min1 <= min2` (hai số nhỏ nhất).

Cập nhật "top-3" bằng chuỗi `if/elif` – **dời xuống** khi có số mới lớn hơn:

```python
if x > max1:   max3, max2, max1 = max2, max1, x
elif x > max2: max3, max2 = max2, x
elif x > max3: max3 = x
```

Phép gán nhiều biến cùng lúc `a, b = b, a` tính **toàn bộ vế phải trước** rồi mới gán – nhờ vậy không bị mất giá trị cũ.

In [9]:
p, q = 1, 2
p, q = q, p          # hoán đổi không cần biến tạm
print(p, q)

2 1


In [10]:
def max_product_one_pass(nums):
    max1 = max2 = max3 = -math.inf
    min1 = min2 = math.inf
    for x in nums:
        # cập nhật 3 lớn nhất
        if x > max1:
            max3, max2, max1 = max2, max1, x
        elif x > max2:
            max3, max2 = max2, x
        elif x > max3:
            max3 = x
        # cập nhật 2 nhỏ nhất
        if x < min1:
            min2, min1 = min1, x
        elif x < min2:
            min2 = x
    return max(max1 * max2 * max3, min1 * min2 * max1)

for nums in ([1, 2, -3], [1, 2, 3, 4], [-1, -2, -3], [-10, -10, 1, 3, 2]):
    print(nums, "->", max_product_one_pass(nums))

[1, 2, -3] -> -6
[1, 2, 3, 4] -> 24
[-1, -2, -3] -> -6
[-10, -10, 1, 3, 2] -> 300


Chú ý: phần cập nhật max và min là hai khối `if` **riêng biệt** (không dùng `elif` nối nhau), vì một số có thể vừa vào top-3 lớn vừa vào top-2 nhỏ (ví dụ mảng chỉ có 3 phần tử).

### 6.1. TRACE một lượt với `[-10, -10, 1, 3, 2]`

| x | max1 | max2 | max3 | min1 | min2 |
|---:|---:|---:|---:|---:|---:|
| -10 | -10 | -inf | -inf | -10 | inf |
| -10 | -10 | -10 | -inf | -10 | -10 |
| 1 | 1 | -10 | -10 | -10 | -10 |
| 3 | 3 | 1 | -10 | -10 | -10 |
| 2 | 3 | 2 | 1 | -10 | -10 |

Cuối: `3·2·1 = 6`, `(-10)(-10)·3 = 300` → **300**.

In [11]:
def trace_one_pass(nums):
    max1 = max2 = max3 = -math.inf
    min1 = min2 = math.inf
    for x in nums:
        if x > max1:
            max3, max2, max1 = max2, max1, x
        elif x > max2:
            max3, max2 = max2, x
        elif x > max3:
            max3 = x
        if x < min1:
            min2, min1 = min1, x
        elif x < min2:
            min2 = x
        print(f"x={x:>4} | max1={max1:>5} max2={max2:>5} max3={max3:>5} | min1={min1:>5} min2={min2:>5}")
    c1, c2 = max1 * max2 * max3, min1 * min2 * max1
    print(f"3 lớn nhất: {c1} | 2 nhỏ × max: {c2} | đáp án: {max(c1, c2)}")
    return max(c1, c2)

trace_one_pass([-10, -10, 1, 3, 2])

x= -10 | max1=  -10 max2= -inf max3= -inf | min1=  -10 min2=  inf
x= -10 | max1=  -10 max2=  -10 max3= -inf | min1=  -10 min2=  -10
x=   1 | max1=    1 max2=  -10 max3=  -10 | min1=  -10 min2=  -10
x=   3 | max1=    3 max2=    1 max3=  -10 | min1=  -10 min2=  -10
x=   2 | max1=    3 max2=    2 max3=    1 | min1=  -10 min2=  -10
3 lớn nhất: 6 | 2 nhỏ × max: 300 | đáp án: 300


300

## 7. Cách 4 — `heapq.nlargest` / `nsmallest` (gọn, `O(n)` với k cố định)

Module chuẩn `heapq` có `nlargest(k, iterable)` và `nsmallest(k, iterable)` trả về `k` phần tử lớn/nhỏ nhất (đã sắp xếp), chạy `O(n log k)` – với `k = 3` coi như `O(n)`.

In [12]:
import heapq

print(heapq.nlargest(3, [-10, -10, 1, 3, 2]), heapq.nsmallest(2, [-10, -10, 1, 3, 2]))

def max_product_heapq(nums):
    big = heapq.nlargest(3, nums)        # [lớn nhất, nhì, ba]
    small = heapq.nsmallest(2, nums)     # [nhỏ nhất, nhì]
    return max(big[0] * big[1] * big[2], small[0] * small[1] * big[0])

print(max_product_heapq([-10, -10, 1, 3, 2]), max_product_heapq([-1, -2, -3]))

[3, 2, 1] [-10, -10]
300 -6


### 7.1. So sánh thời gian thực tế (tham khảo)
`time.perf_counter()` đo thời gian. Số đo tùy máy và mỗi lần chạy; với `n = 10^4` cả ba cách đều chỉ mất vài mili-giây. Khác biệt `O(n log n)` và `O(n)` chỉ rõ khi `n` rất lớn – còn `sorted` được viết bằng C nên hằng số nhỏ, vì vậy bản sắp xếp hoàn toàn đủ tốt để nộp.

In [13]:
import time
import random
rng_t = random.Random(1)
data = [rng_t.randint(-1000, 1000) for _ in range(10_000)]
for name, f in [("sort", max_product_sort), ("one_pass", max_product_one_pass), ("heapq", max_product_heapq)]:
    t0 = time.perf_counter()
    for _ in range(20):
        res = f(data)
    print(f"{name:<9} kết quả={res}  thời gian 20 lần ≈ {time.perf_counter() - t0:.4f}s")

sort      kết quả=1000000000  thời gian 20 lần ≈ 0.0281s
one_pass  kết quả=1000000000  thời gian 20 lần ≈ 0.0139s
heapq     kết quả=1000000000  thời gian 20 lần ≈ 0.0080s


## 8. Code nộp LeetCode

Bản chính: sắp xếp (ngắn, ít lỗi). Bản tối ưu `O(n)` ở ngay sau.

In [14]:
class Solution:
    def maximumProduct(self, nums: list[int]) -> int:
        nums = sorted(nums)
        return max(nums[-1] * nums[-2] * nums[-3], nums[0] * nums[1] * nums[-1])


sol = Solution()
assert sol.maximumProduct([1, 2, -3]) == -6
assert sol.maximumProduct([1, 2, 3, 4]) == 24
assert sol.maximumProduct([-1, -2, -3]) == -6
print("Solution (sort) PASS ví dụ đề")

Solution (sort) PASS ví dụ đề


In [15]:
class SolutionOnePass:
    def maximumProduct(self, nums: list[int]) -> int:
        max1 = max2 = max3 = float("-inf")
        min1 = min2 = float("inf")
        for x in nums:
            if x > max1:
                max3, max2, max1 = max2, max1, x
            elif x > max2:
                max3, max2 = max2, x
            elif x > max3:
                max3 = x
            if x < min1:
                min2, min1 = min1, x
            elif x < min2:
                min2 = x
        return max(max1 * max2 * max3, min1 * min2 * max1)


assert SolutionOnePass().maximumProduct([-10, -10, 1, 3, 2]) == 300
print("SolutionOnePass PASS")

SolutionOnePass PASS


Lưu ý về `float("-inf")`: với `n >= 3`, sau vòng lặp cả 5 biến đều đã là số thật (không còn vô cực), nên tích là số nguyên. Nếu `n < 3` sẽ xuất hiện `inf` trong tích – nhưng constraints bảo đảm `n >= 3`.

## 9. Vì sao đúng? (chứng minh gọn)

Gọi `a` là mảng đã sắp xếp, `P` là tích lớn nhất, `T` là một bộ ba đạt `P`.

**Trường hợp 1 — `a[-1] < 0` (toàn số âm):** mọi tích ba số đều âm; tích âm lớn nhất là tích có **trị tuyệt đối nhỏ nhất**, tức ba số gần 0 nhất `a[-3], a[-2], a[-1]` → ứng viên 1.

**Trường hợp 2 — `a[-1] >= 0`:** trước hết `P >= 0`. Thật vậy: nếu `a[-2], a[-3] >= 0` thì ứng viên 1 `>= 0`; ngược lại mảng có ít nhất hai số âm nên `a[0], a[1] < 0` và ứng viên 2 `= a[0]·a[1]·a[-1] >= 0`. Như vậy luôn có một ứng viên `>= 0`; nếu `P = 0` thì xong.
Nếu `P > 0`, bộ `T` có **0 hoặc 2** số âm:
- **0 số âm:** ba số dương; `a[-1], a[-2], a[-3]` lớn hơn hoặc bằng từng số tương ứng → ứng viên 1 `>= P`.
- **2 số âm `u, v` và một số dương `w`:** `|u|·|v| <= |a[0]|·|a[1]|` (hai số âm nhất) và `w <= a[-1]` → ứng viên 2 `>= P`.

Do đó `max(ứng viên 1, ứng viên 2) = P`. Cách một lượt tính đúng 5 giá trị `a[0], a[1], a[-3], a[-2], a[-1]` nên cho cùng kết quả; mục 11 còn đối chiếu với brute force trên 500 mảng ngẫu nhiên.

## 10. Time / Space Complexity và so sánh

`n` = độ dài mảng.

| Cách | Time | Space | Ghi chú |
|---|---|---|---|
| 1. Brute force | `O(n³)` | `O(1)` | Chỉ làm oracle |
| 2. Sắp xếp | `O(n log n)` | `O(n)` (`sorted` tạo list mới; Timsort) | Ngắn, khó sai |
| 3. Một lượt | `O(n)` | `O(1)` | Tối ưu, nhiều nhánh `if` |
| 4. `heapq` | `O(n log 3) = O(n)` | `O(1)` phụ | Gọn, cần biết `heapq` |

## 11. Bộ test: mẫu đề, biên, đối chiếu ngẫu nhiên

In [16]:
import random

methods = {
    "bruteforce": max_product_bruteforce,
    "sort": max_product_sort,
    "one_pass": max_product_one_pass,
    "heapq": max_product_heapq,
    "Solution": Solution().maximumProduct,
    "SolutionOnePass": SolutionOnePass().maximumProduct,
}
cases = [
    ("đề 1", [1, 2, -3], -6),
    ("đề 2", [1, 2, 3, 4], 24),
    ("đề 3 toàn âm", [-1, -2, -3], -6),
    ("hai âm lớn", [-10, -10, 1, 3, 2], 300),
    ("toàn âm dài", [-5, -4, -3, -2, -1], -6),
    ("có số 0", [-4, -3, -2, 0], 0),
    ("toàn 0", [0, 0, 0], 0),
    ("biên giá trị", [1000, 1000, 1000, -1000], 10**9),
    ("toàn âm, có số rất nhỏ", [-1000, -1000, -1, -2], -2000),
    ("trùng lặp", [2, 2, 2, 2], 8),
]
for name, nums, expected in cases:
    for mname, f in methods.items():
        got = f(nums)
        assert got == expected, (name, mname, got)
    print(f"{name:<18} PASS -> {expected}")

rng = random.Random(628)
for _ in range(500):
    nums = [rng.randint(-6, 6) for _ in range(rng.randint(3, 9))]
    expected = max_product_bruteforce(nums)
    for mname, f in methods.items():
        assert f(nums) == expected, (nums, mname)
print("500 mảng ngẫu nhiên: 6 cách khớp brute force — PASS")

đề 1               PASS -> -6
đề 2               PASS -> 24
đề 3 toàn âm       PASS -> -6
hai âm lớn         PASS -> 300
toàn âm dài        PASS -> -6
có số 0            PASS -> 0
toàn 0             PASS -> 0
biên giá trị       PASS -> 1000000000
toàn âm, có số rất nhỏ PASS -> -2000
trùng lặp          PASS -> 8
500 mảng ngẫu nhiên: 6 cách khớp brute force — PASS


## 12. Lỗi người mới hay gặp

### Lỗi 1: Chỉ lấy ba số lớn nhất
Sai với `[-10,-10,1,3,2]` (ra 6 thay vì 300).

### Lỗi 2: Lấy hai số nhỏ nhất nhân **số lớn thứ ba**/số bất kỳ
Phải nhân với `a[-1]` (lớn nhất).

### Lỗi 3: Khởi tạo `best = 0` trong brute force
Với `[-1,-2,-3]` đáp án `-6` nhưng `best` vẫn là `0`.

### Lỗi 4: Ở cách một lượt, dùng `elif` nối khối min vào khối max
Một số đã vào nhánh max thì không được xét min → sai khi mảng ngắn hoặc giá trị lặp.

In [17]:
def wrong_top3_only(nums):
    a = sorted(nums)
    return a[-1] * a[-2] * a[-3]

def wrong_best_zero(nums):
    best = 0
    for c in combinations(nums, 3):
        best = max(best, math.prod(c))
    return best

def wrong_elif_chain(nums):
    max1 = max2 = max3 = -math.inf
    min1 = min2 = math.inf
    for x in nums:
        if x > max1:
            max3, max2, max1 = max2, max1, x
        elif x > max2:
            max3, max2 = max2, x
        elif x > max3:
            max3 = x
        elif x < min1:                      # SAI: nối vào chuỗi max
            min2, min1 = min1, x
        elif x < min2:
            min2 = x
    return max(max1 * max2 * max3, min1 * min2 * max1)

print("Chỉ 3 lớn nhất  :", wrong_top3_only([-10, -10, 1, 3, 2]), "(đúng 300)")
print("best = 0        :", wrong_best_zero([-1, -2, -3]), "(đúng -6)")
print("elif nối chuỗi  :", wrong_elif_chain([-10, -10, 1, 3, 2]), "(đúng 300)")

Chỉ 3 lớn nhất  : 6 (đúng 300)
best = 0        : 0 (đúng -6)
elif nối chuỗi  : inf (đúng 300)


## 13. Mẹo nhận dạng pattern

- Bài "tích lớn nhất/nhỏ nhất" → luôn nghĩ tới **dấu**: hai âm thành dương; số 0 "xóa" tích.
- Chỉ cần vài phần tử **cực trị** (lớn nhất/nhỏ nhất) → không cần sắp xếp hết: giữ **top-k** trong một lượt hoặc dùng `heapq`.
- Khi nghi ngờ các "ứng viên" có đủ không → viết brute force nhỏ và **đối chiếu ngẫu nhiên** (như mục 11).

**Bài liên quan:** 152. Maximum Product Subarray (giữ cả max và min vì số âm đảo dấu), 1464. Maximum Product of Two Elements in an Array, 2367/… các bài chọn k phần tử.

## 14. Biến thể

**Biến thể A — Tích lớn nhất của hai số:** ứng viên `a[-1]·a[-2]` và `a[0]·a[1]`.

**Biến thể B — Tích nhỏ nhất của ba số:** đối xứng: `a[0]·a[1]·a[2]` hoặc `a[0]·a[-1]·a[-2]`.

**Biến thể C — Tổng quát `k` số:** sau khi sắp xếp, bộ tối ưu luôn gồm `j` số nhỏ nhất và `k − j` số lớn nhất với một `j` nào đó (`0 <= j <= k`) → thử hết `k + 1` khả năng. (Đã kiểm chứng bằng brute force – xem đáp án bài 6.)

In [18]:
def max_product_two(nums):
    a = sorted(nums)
    return max(a[-1] * a[-2], a[0] * a[1])

def min_product_three(nums):
    a = sorted(nums)
    return min(a[0] * a[1] * a[2], a[0] * a[-1] * a[-2])

print("A:", max_product_two([-10, -3, 1, 2]), "| B:", min_product_three([-10, -3, 1, 2, 5]))

A: 30 | B: -100


## 15. Bài tập (TODO – không gọi khi chưa làm)

**Bài 1 — Tính tay.** `nums = [-8, -7, 0, 1, 6, 9]`: tính hai ứng viên và đáp án.

**Bài 2 — Đoán output.** `sorted([3, -1, -5, 2])[-3]` là gì?

**Bài 3 — Viết lại** `maximumProduct` bằng sắp xếp.

**Bài 4 — Trả về chính bộ ba** (dạng list đã sắp xếp tăng dần) cho tích lớn nhất.

**Bài 5 — Biến thể B:** tích **nhỏ nhất** của ba số, chỉ một lượt duyệt (giữ 3 min + 2 max).

**Bài 6 — Mini challenge:** `max_product_k(nums, k)` tổng quát; kiểm tra bằng brute force với `combinations`.

In [19]:
answer_ex1 = None     # TODO: (ứng viên 1, ứng viên 2, đáp án)
answer_ex2 = None     # TODO


def max_product_exercise(nums):
    # TODO: sorted, hai ứng viên, max
    pass


def best_triple(nums):
    # TODO: trả về list 3 số đã sắp xếp
    pass

# Kiểm tra sau khi làm:
# assert max_product_exercise([-10, -10, 1, 3, 2]) == 300
# assert best_triple([-10, -10, 1, 3, 2]) == [-10, -10, 3]

In [20]:
def min_product_three_one_pass(nums):
    # TODO: min1 <= min2 <= min3, max1 >= max2
    pass


def max_product_k(nums, k):
    # TODO: sort; thử j số nhỏ nhất + (k - j) số lớn nhất
    pass

# Kiểm tra sau khi làm:
# assert min_product_three_one_pass([-10, -3, 1, 2, 5]) == -100
# assert max_product_k([-10, -10, 1, 3, 2], 3) == 300

## 16. Gợi ý 3 mức

**Bài 1:** (1) Sắp xếp sẵn rồi. (2) `9·6·1` và `(-8)(-7)·9`. (3) So sánh 54 và 504.

**Bài 2:** (1) Sắp xếp: `[-5, -1, 2, 3]`. (2) `[-3]` là phần tử thứ ba từ cuối. (3) Đếm ngược: 3, 2, -1.

**Bài 3:** (1) Hai ứng viên. (2) `a[-1]*a[-2]*a[-3]`, `a[0]*a[1]*a[-1]`. (3) `max(...)`.

**Bài 4:** (1) Giữ cả bộ ba cùng tích. (2) So sánh tích rồi chọn list tương ứng. (3) Trả `sorted(...)`.

**Bài 5:** (1) Đối xứng với cách một lượt. (2) Ứng viên: `min1·min2·min3` và `min1·max1·max2`. (3) Hai khối `if` riêng cho min-3 và max-2.

**Bài 6:** (1) Sắp xếp. (2) Với `j` từ `0` tới `k`: `a[:j] + a[n-(k-j):]`. (3) Cẩn thận `k - j = 0` → `a[n:]` là list rỗng; dùng `math.prod`.

## 17. Đáp án tham khảo (chạy kiểm tra)

In [21]:
a1 = sorted([-8, -7, 0, 1, 6, 9])
ans_ex1 = (a1[-1] * a1[-2] * a1[-3], a1[0] * a1[1] * a1[-1], max_product_sort(a1))   # (54, 504, 504)
ans_ex2 = sorted([3, -1, -5, 2])[-3]                                                  # -1


def ans_max_product(nums):
    a = sorted(nums)
    return max(a[-1] * a[-2] * a[-3], a[0] * a[1] * a[-1])


def ans_best_triple(nums):
    a = sorted(nums)
    t1, t2 = [a[-3], a[-2], a[-1]], [a[0], a[1], a[-1]]
    return sorted(t1) if math.prod(t1) >= math.prod(t2) else sorted(t2)


def ans_min_product_three_one_pass(nums):
    min1 = min2 = min3 = math.inf
    max1 = max2 = -math.inf
    for x in nums:
        if x < min1:
            min3, min2, min1 = min2, min1, x
        elif x < min2:
            min3, min2 = min2, x
        elif x < min3:
            min3 = x
        if x > max1:
            max2, max1 = max1, x
        elif x > max2:
            max2 = x
    return min(min1 * min2 * min3, min1 * max1 * max2)


def ans_max_product_k(nums, k):
    a = sorted(nums)
    n = len(a)
    return max(math.prod(a[:j] + a[n - (k - j):]) for j in range(k + 1))


print("Bài 1:", ans_ex1, "| Bài 2:", ans_ex2)
assert ans_ex1 == (54, 504, 504) and ans_ex2 == -1
assert ans_max_product([-10, -10, 1, 3, 2]) == 300
assert ans_best_triple([-10, -10, 1, 3, 2]) == [-10, -10, 3]
assert ans_best_triple([1, 2, 3, 4]) == [2, 3, 4]
assert ans_min_product_three_one_pass([-10, -3, 1, 2, 5]) == -100
assert ans_max_product_k([-10, -10, 1, 3, 2], 3) == 300

rng = random.Random(6280)
for _ in range(500):
    nums = [rng.randint(-6, 6) for _ in range(rng.randint(3, 8))]
    k = rng.randint(1, len(nums))
    assert ans_max_product_k(nums, k) == max(math.prod(c) for c in combinations(nums, k)), (nums, k)
    assert ans_min_product_three_one_pass(nums) == min(math.prod(c) for c in combinations(nums, 3)), nums
    assert math.prod(ans_best_triple(nums)) == max_product_bruteforce(nums)
print("Đáp án: tất cả test PASS (kể cả 500 lần đối chiếu brute force cho k tổng quát)")

Bài 1: (54, 504, 504) | Bài 2: -1
Đáp án: tất cả test PASS (kể cả 500 lần đối chiếu brute force cho k tổng quát)


## 18. Template nhớ nhanh và checklist submit

```python
class Solution:
    def maximumProduct(self, nums: list[int]) -> int:
        a = sorted(nums)
        return max(a[-1] * a[-2] * a[-3],    # ba số lớn nhất
                   a[0] * a[1] * a[-1])      # hai số âm "to" × số lớn nhất
```

Câu nhớ: **"ba lớn nhất, hoặc hai nhỏ nhất nhân lớn nhất"**.

### Checklist
- [ ] Đã xét trường hợp **hai số âm** lớn trị tuyệt đối?
- [ ] Ứng viên 2 nhân với `a[-1]` (lớn nhất), không phải số khác?
- [ ] Không khởi tạo max bằng `0` (đáp án có thể âm)?
- [ ] Bản một lượt: hai khối `if` max/min **tách riêng**?
- [ ] Đã thử: toàn dương, toàn âm, có 0, đúng 3 phần tử, giá trị ±1000?